# SPY 0DTE Flow-Following Strategy

Scan the 0DTE option trade tape for **large BUY-aggressor prints** (institutional 'smart money'). Copy the trade: buy the same option, manage with PT / SL. One trade/day, first qualifying print wins. **No time-based exit** — trades run until profit target, stop loss, or 3:55 PM ET (hard close).

**Aggressor classification** (rough proxy without NBBO quotes):
- `trade_price >= bar_open` → buy aggressor → follow
- `trade_price <  bar_open` → sell aggressor → skip

**Multi-leg filter (new):** Prints whose Polygon/OPRA condition codes indicate they're part of a spread / vertical / risk-reversal / stock-tied combo are skipped. The code set lives in `config.MULTI_LEG_CONDITION_CODES`; the StrategyParams field is `exclude_multi_leg=True` by default. Each day's sweep logs how many large prints got filtered (visible with `-v`).

**Caveats to keep in mind:**
- The condition-code list is based on the documented OPRA reference but may be slightly off in either direction. If results look strange, the codes can be adjusted.
- Aggressor proxy is coarser than commercial flow services that use NBBO quotes.
- At the higher size thresholds (4500–5000) signal frequency drops sharply. After multi-leg filtering, it'll drop further.

**Sweep:** 150 configs (5 size × 3 PT × 5 SL × 2 entry modes).
1. **7 days** (cell 8) — sanity check, ~1 min.
2. **30 days** (cell 10) — viability check, ~5 min. Trade-tape cache from prior run is reused.
3. **365 days** (cell 12) — full year. Trade tapes are cached from prior runs.

## 1. Setup

In [ ]:
import os, sys
REPO = '/content/iron-condor'
BRANCH = 'claude/spy-options-trading-bot-ri4Ms'
if not os.path.isdir(REPO):
    !git clone --quiet -b {BRANCH} https://github.com/coolin815/iron-condor.git {REPO}
else:
    !cd {REPO} && git pull --quiet
SRC = REPO + '/src'
os.environ['PYTHONPATH'] = SRC + os.pathsep + os.environ.get('PYTHONPATH', '')
if SRC not in sys.path:
    sys.path.insert(0, SRC)
os.chdir(REPO)
print('OK')

## 2. Paste your Polygon key

In [ ]:
import os, getpass
os.environ['POLYGON_API_KEY'] = getpass.getpass('Polygon API key: ')
print('Key loaded:', len(os.environ['POLYGON_API_KEY']), 'chars')

## 3. Smoke test — one recent day
Default size threshold 1500 contracts. Confirms trade-tape fetch + aggressor classification + entry pipeline.

In [ ]:
!python -m iron_condor.cli --smoke -v

## 4. Quick sweep — past 7 days
First validation. ~5–10 min. Just want to see if any trades trigger and the pipeline runs end-to-end. With only 5 trading days, win-rate and return numbers are mostly noise — we're checking the code path, not the strategy yet.

In [ ]:
!python -m iron_condor.cli --days 7 --sweep

## 4b. Medium sweep — past 30 days
~20–30 min. Now ~21 trading days — enough to start spotting whether top configs cluster around realistic returns or are riding 1–2 lucky trades. Trade-tape cache from cell 8 is reused.

In [ ]:
!python -m iron_condor.cli --days 30 --sweep

## 4c. Full year — optional, only if 30 days looked promising
**3–6 hours.** Trade-tape fetches per strike per day, paginated, throttled. Re-uses the 30-day cache so it's incremental — but ~330 new days × ~10 strikes × paginated fetches is the bulk of the work.

If cells 8/10 already showed the strategy doesn't have edge, skip this and don't burn API quota.

In [ ]:
!python -m iron_condor.cli --days 365 --sweep

## 5. Top 20 configs
Reads `results/sweep_summary.csv` written by whichever sweep cell you last ran (7d, 30d, or 365d).

In [ ]:
import pandas as pd
summary = pd.read_csv('results/sweep_summary.csv')
summary.head(20)

## 6. Size-threshold breakdown — does bigger = better?
Per-threshold average return, win rate, trade count.
Useful to see if 'bigger prints = stronger signal' actually holds in the data.

In [ ]:
import re
rows = []
for _, r in summary.iterrows():
    m = re.match(r'sz(\d+)\|pt(\d+)\|sl(\d+)', r['config'])
    if m:
        rows.append({
            'sz': int(m.group(1)),
            'pt': int(m.group(2)),
            'sl': int(m.group(3)),
            'return': r['total_return_pct'],
            'win_rate': r['win_rate'],
            'trades': r['n_trades'],
        })
grid = pd.DataFrame(rows)
print('=== Avg return % by size threshold ===')
print(grid.groupby('sz')['return'].agg(['mean', 'median', 'max']).round(1))
print('\n=== Avg win rate by size threshold ===')
print(grid.groupby('sz')['win_rate'].agg(['mean', 'median', 'max']).round(2))
print('\n=== Trade count by size threshold ===')
print(grid.groupby('sz')['trades'].agg(['mean', 'median', 'min', 'max']).round(0))

## 6b. Entry-mode comparison — does instant beat next-bar?
If the print is genuinely informative, instant entries should win — we capture the directional move that next-bar entries miss in the first 60 sec.

In [ ]:
for em_short, label in [('inst', 'instant'), ('next', 'next_bar_open')]:
    s = summary[summary['config'].str.contains(f'em={em_short}')]
    if not s.empty:
        best = s.iloc[0]
        print(f'\n=== Best {label.upper()} config ===')
        print(f"  {best['config']}")
        print(f"  return={best['total_return_pct']:>+8.1f}%")
        print(f"  win_rate={best['win_rate']:>6.1%}")
        print(f"  trades={best['n_trades']}")
        print(f"  avg_pnl=${best['avg_net_pnl']:>+8.2f}")

## 7. Top config breakdown — call vs put, exit reasons

In [ ]:
trades = pd.read_csv('results/sweep_trades.csv')
top_cfg = summary.iloc[0]['config']
sub = trades[trades['config'] == top_cfg]
taken = sub[sub['exit_reason'].isin(['profit', 'stop', 'time_stop'])]
print(f'Top config: {top_cfg}')
print(f'Total days: {len(sub)}, taken trades: {len(taken)}')
print('\n=== Call vs put performance ===')
if not taken.empty and 'right' in taken.columns:
    print(taken.groupby('right').agg(
        trades=('net_pnl', 'count'),
        win_rate=('net_pnl', lambda s: (s > 0).mean()),
        avg_pnl=('net_pnl', 'mean'),
        total_pnl=('net_pnl', 'sum'),
    ).round(2))
print('\n=== Exit reasons ===')
print(sub['exit_reason'].value_counts())
print('\n=== Print sizes captured (taken trades only) ===')
if not taken.empty and 'print_size' in taken.columns:
    print(taken['print_size'].describe().round(0))

## 8. Equity curve — top 3 configs

In [ ]:
import matplotlib.pyplot as plt
top3 = summary.head(3)['config'].tolist()
fig, ax = plt.subplots(figsize=(11, 5))
for cfg in top3:
    sub = trades[trades['config'] == cfg].sort_values('day')
    ax.plot(pd.to_datetime(sub['day']), sub['balance_after'], label=cfg, linewidth=1.5)
ax.axhline(1500, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Date'); ax.set_ylabel('Balance ($)')
ax.set_title('Flow-following — top 3 equity curves')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 9. Per-month breakdown — was 30-day a regime or a fluke?
Splits the year by calendar month. Two views:
1. **Aggregate across all configs** — does any month look genuinely different (high WR across the board)? That's a regime signature.
2. **Top config monthly** — was the winner's edge spread evenly over the year, or did it earn it all in 1–2 months?

In [ ]:
trades_all = pd.read_csv('results/sweep_trades.csv')
trades_all['day'] = pd.to_datetime(trades_all['day'])
trades_all['month'] = trades_all['day'].dt.to_period('M').astype(str)
taken_all = trades_all[trades_all['exit_reason'].isin(['profit', 'stop', 'time_stop'])]

print('=== Per-month aggregate (all configs averaged) ===')
monthly = taken_all.groupby('month').agg(
    trades=('net_pnl', 'count'),
    win_rate=('net_pnl', lambda s: (s > 0).mean()),
    avg_pnl=('net_pnl', 'mean'),
    total_pnl=('net_pnl', 'sum'),
).round(2)
print(monthly.to_string())

print(f'\n=== Top config ({summary.iloc[0]["config"]}) per month ===')
top_cfg = summary.iloc[0]['config']
top_taken = taken_all[taken_all['config'] == top_cfg]
if not top_taken.empty:
    top_monthly = top_taken.groupby('month').agg(
        trades=('net_pnl', 'count'),
        win_rate=('net_pnl', lambda s: (s > 0).mean()),
        avg_pnl=('net_pnl', 'mean'),
        total_pnl=('net_pnl', 'sum'),
    ).round(2)
    print(top_monthly.to_string())
else:
    print('(no taken trades for top config)')